In [1]:
import os
os.chdir(r'C:\Users\HP\Desktop\rag-complaint-chatbot')
print("Working directory:", os.getcwd())

Working directory: C:\Users\HP\Desktop\rag-complaint-chatbot


In [ ]:
# ============================================================
# Task 2: Text Chunking, Embedding, and Vector Store Indexing
# CrediTrust Financial - RAG Complaint Chatbot
# ============================================================

import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
import pickle

# ── 1. LOAD CLEANED DATA ────────────────────────────────────
print("Loading cleaned dataset...")

df = pd.read_csv("data/processed/filtered_complaints.csv")
print(f"Loaded {len(df)} complaints")
print(f"\nProduct category distribution:")
print(df['product_category'].value_counts())


# ── 2. STRATIFIED SAMPLE ────────────────────────────────────
print("\n============ STRATIFIED SAMPLING ============")

SAMPLE_SIZE = 12000

def stratified_sample(df, total_size, category_col):
    categories = df[category_col].value_counts()
    proportions = categories / len(df)
    samples = []
    for category, proportion in proportions.items():
        n = int(total_size * proportion)
        cat_df = df[df[category_col] == category]
        n = min(n, len(cat_df))
        sample = cat_df.sample(n=n, random_state=42)
        samples.append(sample)
    result = pd.concat(samples).reset_index(drop=True)
    return result

df_sample = stratified_sample(df, SAMPLE_SIZE, 'product_category')
print(f"\nSample size: {len(df_sample)}")
print(f"\nSample distribution:")
print(df_sample['product_category'].value_counts())


# ── 3. TEXT CHUNKING ────────────────────────────────────────
print("\n============ CHUNKING ============")

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    chunks = []
    start = 0
    text = text.strip()
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if len(chunk.strip()) > 50:
            chunks.append(chunk.strip())
        start = end - overlap
        if start >= len(text):
            break
    return chunks

print("Chunking narratives...")
all_chunks = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    text = str(row['cleaned_narrative'])
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'chunk_text': chunk,
            'complaint_id': row.get('Complaint ID', ''),
            'product_category': row['product_category'],
            'product': row.get('Product', ''),
            'issue': row.get('Issue', ''),
            'chunk_index': i,
            'total_chunks': len(chunks)
        })

print(f"\nTotal chunks created: {len(all_chunks)}")
chunks_df = pd.DataFrame(all_chunks)
print(f"\nChunks per product category:")
print(chunks_df['product_category'].value_counts())


# ── 4. GENERATE EMBEDDINGS ──────────────────────────────────
print("\n============ EMBEDDING ============")
print("Loading embedding model (all-MiniLM-L6-v2)...")

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
texts = chunks_df['chunk_text'].tolist()

print(f"Generating embeddings for {len(texts)} chunks...")
print("This may take several minutes...")

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")


# ── 5. BUILD FAISS INDEX ────────────────────────────────────
print("\n============ BUILDING FAISS INDEX ============")

os.makedirs("vector_store", exist_ok=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"FAISS index built with {index.ntotal} vectors")

faiss.write_index(index, "vector_store/complaints.index")
print("Saved: vector_store/complaints.index")

chunks_df.to_csv("vector_store/chunks_metadata.csv", index=False)
print("Saved: vector_store/chunks_metadata.csv")

np.save("vector_store/embeddings.npy", embeddings)
print("Saved: vector_store/embeddings.npy")

print("\n✅ Task 2 Complete!")

Loading cleaned dataset...
Loaded 445573 complaints

Product category distribution:
product_category
Credit Card        189334
Savings Account    140317
Money Transfer      98684
Personal Loan       17238
Name: count, dtype: int64

============ STRATIFIED SAMPLING ============

Sample size: 11998

Sample distribution:
product_category
Credit Card        5099
Savings Account    3778
Money Transfer     2657
Personal Loan       464
Name: count, dtype: int64

============ CHUNKING ============
Chunking narratives...


100%|██████████| 11998/11998 [00:00<00:00, 12563.51it/s]



Total chunks created: 34058

Chunks per product category:
product_category
Credit Card        14783
Savings Account    11108
Money Transfer      6896
Personal Loan       1271
Name: count, dtype: int64

============ EMBEDDING ============
Loading embedding model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\HP\Desktop\rag-complaint-chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]